# Multi-Period Optimization

Multi-period optimization extends the model with a **period** dimension
(e.g. 2025, 2030, 2035). Each period shares the same timestep structure
but can have independent dispatch and sizing decisions.

Period weights scale the objective — they control how much each period
"counts". The library provides **weight slots**; you decide the numbers.

Common choices:

| Strategy | Weights for `[2025, 2030, 2035]` | Use case |
|---|---|---|
| **Flat (default)** | `[5, 5, 5]` (inferred from gaps) | Equal importance per year |
| **NPV** | `[4.55, 3.56, 2.79]` (discounted) | Present-value costing |
| **Custom** | any `list[float]` | Scenario-specific |

This notebook shows how to use `period_weights_periodic` on an `Effect`
to apply NPV discounting to costs while keeping CO₂ on flat weights.

In [ ]:
from datetime import datetime

import numpy as np
import pandas as pd

from fluxopt import Carrier, Converter, Effect, Flow, Port, Sizing, optimize

## System

A factory needs heat supplied by a gas boiler whose capacity is optimized.

```
Gas Grid ──► [gas] ──► Boiler η=90% ──► [heat] ──► Factory
 0.05 €/kWh                                       demand profile
 0.2 kg CO₂/kWh
```

Three planning periods (2025, 2030, 2035), each representing 5 years.
Four representative timesteps per period.

In [ ]:
timesteps = [datetime(2025, 1, 15, h) for h in range(6, 10)]
periods = [2025, 2030, 2035]
demand_profile = [0.6, 1.0, 0.8, 0.5]  # relative to size=100 kW

## Flat weights (default)

When no weights are specified, they are inferred from the gaps between
period labels — here `[5, 5, 5]`. This treats every year equally.

In [ ]:
def build_system(**effect_kw):
    """Build the system with configurable Effect parameters."""
    return {
        'timesteps': timesteps,
        'carriers': [Carrier('gas'), Carrier('heat')],
        'effects': [
            Effect('cost', unit='EUR', is_objective=True, **effect_kw),
            Effect('CO2', unit='kg'),
        ],
        'ports': [
            Port(
                'gas_grid',
                imports=[
                    Flow('gas', size=1000, effects_per_flow_hour={'cost': 0.05, 'CO2': 0.2}),
                ],
            ),
            Port(
                'factory',
                exports=[
                    Flow('heat', size=100, fixed_relative_profile=demand_profile),
                ],
            ),
        ],
        'converters': [
            Converter.boiler(
                'boiler',
                thermal_efficiency=0.9,
                fuel_flow=Flow('gas', size=Sizing(50, 200, effects_per_size={'cost': 10})),
                thermal_flow=Flow('heat', size=200),
            )
        ],
        'periods': periods,
    }


result_flat = optimize(**build_system())
result_flat.effect_totals.sel(effect='cost').to_pandas()

Each period has the same cost (same demand, same system). The objective
sums `weight × total` across periods:

In [ ]:
result_flat.objective  # 5 * total + 5 * total + 5 * total

## NPV weights

Future costs are worth less today. To discount them, compute your own
weight factors and pass them via `Effect.period_weights_periodic`.

With a 5% discount rate and 5-year periods:

In [ ]:
r = 0.05
period_offsets = [0, 5, 10]

# annuity sum for each 5-year block, discounted to year 0
npv_weights = [round(sum(1 / (1 + r) ** (y0 + y) for y in range(5)), 2) for y0 in period_offsets]

pd.DataFrame(
    {
        'period': periods,
        'flat (default)': [5, 5, 5],
        'NPV (r=5%)': npv_weights,
    }
).set_index('period')

Now optimize with NPV weights on cost. CO₂ has no override, so it
keeps the global flat weights — physical quantities should not be
discounted.

In [ ]:
result_npv = optimize(**build_system(period_weights_periodic=npv_weights))

## Comparison

The per-period costs are identical — only the weighting differs:

In [ ]:
totals_flat = result_flat.effect_totals.sel(effect='cost').values
totals_npv = result_npv.effect_totals.sel(effect='cost').values

pd.DataFrame(
    {
        'cost/period': totals_flat.round(2),
        'weight (flat)': [5, 5, 5],
        'objective contribution (flat)': (totals_flat * 5).round(2),
        'weight (NPV)': npv_weights,
        'objective contribution (NPV)': (totals_npv * np.array(npv_weights)).round(2),
    },
    index=pd.Index(periods, name='period'),
)

In [ ]:
pd.Series(
    {
        'Flat (default)': round(result_flat.objective, 2),
        'NPV (r=5%)': round(result_npv.objective, 2),
    },
    name='objective (EUR)',
)

## How it works

The objective function weights each period's total effect:

$$
\min \sum_p \omega^{\text{periodic}}_{k^*,p} \cdot \Phi_{k^*,p}
$$

where $\Phi_{k,p}$ combines temporal (dispatch) and periodic (sizing) costs.

| Parameter | Default | Override | Purpose |
|---|---|---|---|
| `period_weights` | Inferred from gaps | `optimize(period_weights=...)` | Global weights for all effects |
| `period_weights_periodic` | Global weights | `Effect(period_weights_periodic=...)` | Per-effect override (e.g. NPV on cost only) |
| `period_weights_once` | `1` (no scaling) | `Effect(period_weights_once=...)` | One-time cost weighting |

The library provides the weight slots — **you** decide the factors.
Flat, NPV, annuity, or any custom scheme.